# Clase 066 — Curva ROC y AUC

Qué mide la curva **ROC** (TPR vs FPR), cómo calcular el **AUC** y cuándo ROC engaña: en datasets desbalanceados la curva PR es más honesta.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, precision_score, recall_score)

np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
ytr5 = (ytr == 5)
print('prevalencia de 5s en train: {:.3f}'.format(ytr5.mean()))

## 1. Scores y curva ROC

Obtenemos scores out-of-fold del `SGDClassifier` con `decision_function` y graficamos TPR vs FPR. La diagonal es el azar.

In [ ]:
sgd = SGDClassifier(random_state=42)
y_scores = cross_val_predict(sgd, Xtr, ytr5, cv=3,
                             method='decision_function', n_jobs=1)
fpr, tpr, thr = roc_curve(ytr5, y_scores)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#37a', label='SGD')
ax.plot([0, 1], [0, 1], 'k--', label='azar')
ax.set_xlabel('FPR (1 - especificidad)')
ax.set_ylabel('TPR (recall)')
ax.set_title('Curva ROC')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. AUC

El AUC resume la curva: 0.5 = azar, 1.0 = perfecto. Es la probabilidad de rankear un positivo por encima de un negativo al azar.

In [ ]:
auc_sgd = roc_auc_score(ytr5, y_scores)
print(f'AUC (SGD): {auc_sgd:.4f}')
assert auc_sgd > 0.9
print('interpretacion: muy por encima de 0.5 (azar), el modelo separa bien las clases.')

## 3. Comparar dos modelos por ROC

Entrenamos un `RandomForestClassifier` (usa `predict_proba`, columna 1) y superponemos ambas curvas ROC. Elegimos por AUC.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
y_proba_rf = cross_val_predict(forest, Xtr, ytr5, cv=3,
                               method='predict_proba', n_jobs=1)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(ytr5, y_proba_rf)
auc_rf = roc_auc_score(ytr5, y_proba_rf)

print(f'AUC SGD:   {auc_sgd:.4f}')
print(f'AUC Forest: {auc_rf:.4f}')
ganador = 'RandomForest' if auc_rf > auc_sgd else 'SGD'
print('gana por AUC:', ganador)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'SGD (AUC={auc_sgd:.3f})', color='#37a')
ax.plot(fpr_rf, tpr_rf, label=f'Forest (AUC={auc_rf:.3f})', color='#3a7')
ax.plot([0, 1], [0, 1], 'k--', label='azar')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC: SGD vs RandomForest')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. ROC vs PR en desbalance extremo

Con un dataset 99/1, la ROC sigue "linda" (FPR bajo por los muchos TN) mientras la curva PR revela la pobreza real. Las graficamos lado a lado.

In [ ]:
Xi, yi = make_classification(n_samples=10000, weights=[0.99, 0.01],
                             n_informative=5, random_state=42)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3,
                                              stratify=yi, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1).fit(Xi_tr, yi_tr)
proba = clf.predict_proba(Xi_te)[:, 1]

auc_imb = roc_auc_score(yi_te, proba)
ap_imb = average_precision_score(yi_te, proba)
print(f'AUC (se ve alto):        {auc_imb:.3f}')
print(f'Average Precision (real): {ap_imb:.3f}')

f_roc, t_roc, _ = roc_curve(yi_te, proba)
p_pr, r_pr, _ = precision_recall_curve(yi_te, proba)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(f_roc, t_roc, color='#37a')
a1.plot([0, 1], [0, 1], 'k--')
a1.set_title(f'ROC (AUC={auc_imb:.3f}) - optimista')
a1.set_xlabel('FPR'); a1.set_ylabel('TPR')
a2.plot(r_pr, p_pr, color='#c33')
a2.set_title(f'PR (AP={ap_imb:.3f}) - realista')
a2.set_xlabel('recall'); a2.set_ylabel('precision')
plt.tight_layout()
plt.show()

assert auc_imb > ap_imb

## 5. Punto operativo (índice de Youden)

Sobre la ROC del SGD, el umbral que maximiza `TPR - FPR` (índice de Youden) da un punto operativo concreto. Reportamos precision y recall ahí.

In [ ]:
youden = tpr - fpr
j = int(np.argmax(youden))
umbral_j = thr[j]
pred_j = (y_scores >= umbral_j)
print(f'umbral Youden: {umbral_j:.3f}')
print(f'TPR-FPR maximo: {youden[j]:.3f}')
print(f'precision: {precision_score(ytr5, pred_j):.3f}')
print(f'recall:    {recall_score(ytr5, pred_j):.3f}')

## Ejercicios

1. **Interpretar el AUC.** Escribí en una línea qué significa un AUC de 0.98 en este problema.
2. **Elegir el ganador.** Entre SGD y RandomForest, ¿cuál elegirías y con qué métrica (AUC o AP), dado que la prevalencia es ~10%?
3. **AUC < 0.5.** Invertí los scores (`-y_scores`) y recalculá el AUC. ¿Qué obtenés y por qué?
4. **Multiclase.** Probá `roc_auc_score(..., multi_class='ovr')` sobre las 10 clases con `predict_proba`. ¿Qué informa?

## Conclusiones

- La **ROC** grafica TPR vs FPR barriendo umbrales; el **AUC** la resume (0.5 azar, 1.0 perfecto).
- El AUC depende del **orden** de los scores, no de su escala: `decision_function` o `predict_proba` dan lo mismo.
- En **desbalance fuerte**, la ROC se ve optimista (muchos TN bajan el FPR); la **curva PR / AP** es más honesta.
- El AUC sirve para **comparar modelos**; para deployar hay que elegir un umbral operativo (ej.: índice de Youden).
- Reportá siempre AUC **y** AP cuando la clase positiva es rara.

## ✅ Soluciones de los ejercicios

Ejercicios del README resueltos sobre el detector 'es un 5' de `load_digits` (offline). Reutilizamos `sgd`, `ytr5`, `y_scores`, `fpr`, `tpr`, `thr`, `auc_sgd` y `forest` ya definidos arriba.

**Ej. 1 — Scores y curva ROC.** Scores out-of-fold del SGD con `decision_function` y curva ROC (TPR vs FPR); la diagonal es el azar.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve

ys = cross_val_predict(sgd, Xtr, ytr5, cv=3, method="decision_function", n_jobs=1)
f, t, th = roc_curve(ytr5, ys)
plt.plot(f, t, label="SGD"); plt.plot([0, 1], [0, 1], "k--", label="azar")
plt.xlabel("FPR"); plt.ylabel("TPR (recall)"); plt.legend(); plt.title("Curva ROC"); plt.show()
assert f[0] == 0 and t[-1] == 1

**Ej. 2 — AUC.** `roc_auc_score`: 0.5 = azar, 1.0 = perfecto. Es la probabilidad de rankear un positivo por encima de un negativo tomados al azar.

In [ ]:

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(ytr5, ys)
print(f"AUC (SGD) = {auc:.4f}")
print("interpretacion: probabilidad de que un 5 al azar reciba mayor score que un no-5 al azar")
assert auc > 0.9

**Ej. 3 — Comparar dos modelos por ROC.** `RandomForestClassifier` no tiene `decision_function`: usamos `predict_proba[:,1]`. Superponemos ambas ROC y elegimos por AUC.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
proba_rf = cross_val_predict(rf, Xtr, ytr5, cv=3, method="predict_proba", n_jobs=1)[:, 1]
f_rf, t_rf, _ = roc_curve(ytr5, proba_rf)
auc_rf = roc_auc_score(ytr5, proba_rf)
auc_sgd_local = roc_auc_score(ytr5, ys)
print(f"AUC SGD={auc_sgd_local:.4f} | AUC Forest={auc_rf:.4f}")
print("gana:", "RandomForest" if auc_rf > auc_sgd_local else "SGD")
plt.plot(f, t, label=f"SGD (AUC={auc_sgd_local:.3f})")
plt.plot(f_rf, t_rf, label=f"Forest (AUC={auc_rf:.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.title("ROC: SGD vs RF"); plt.show()
assert auc_rf > 0.9

**Ej. 4 — ROC vs PR en desbalance.** Con 99/1 la ROC se ve 'linda' (muchos TN mantienen el FPR bajo) mientras la curva PR revela la pobreza real. Las graficamos lado a lado.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

Xi, yi = make_classification(n_samples=10000, weights=[0.99, 0.01], n_informative=5, random_state=42)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1).fit(Xi_tr, yi_tr)
proba = clf.predict_proba(Xi_te)[:, 1]
auc_imb, ap_imb = roc_auc_score(yi_te, proba), average_precision_score(yi_te, proba)
print(f"AUC (optimista)={auc_imb:.3f} | Average Precision (realista)={ap_imb:.3f}")
fr, tr_, _ = roc_curve(yi_te, proba); pp, rr, _ = precision_recall_curve(yi_te, proba)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(fr, tr_); a1.plot([0, 1], [0, 1], "k--"); a1.set_title(f"ROC (AUC={auc_imb:.3f})")
a2.plot(rr, pp, color="#c33"); a2.set_title(f"PR (AP={ap_imb:.3f})"); plt.show()
assert auc_imb > ap_imb, "en desbalance la ROC se ve mejor que la PR"

**Ej. 5 — Punto operativo (índice de Youden).** El umbral que maximiza `TPR - FPR` da un punto operativo concreto; reportamos precision y recall ahí.

In [ ]:

import numpy as np
from sklearn.metrics import precision_score, recall_score

youden = t - f
j = int(np.argmax(youden))
th_j = th[j]
pred_j = (ys >= th_j)
print(f"umbral Youden={th_j:.3f} | TPR-FPR max={youden[j]:.3f}")
print(f"precision={precision_score(ytr5, pred_j):.3f} | recall={recall_score(ytr5, pred_j):.3f}")
assert 0 <= j < len(th)